<a href="https://colab.research.google.com/github/dsaenzd04/IA_PROGRAMACION/blob/main/modelo_bigramas_didactico_corregido.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-modelo de Lenguaje con Bigramas

## Cual es el reto

En este proyecto vamos a construir, paso a paso, un **modelo que genera texto**. Le daremos una palabra inicial (por ejemplo, `'el'`) y el modelo escribira una frase completa palabra a palabra, eligiendo cada nueva palabra **segun la probabilidad** de que aparezca despues de la anterior.

Este es, en esencia, el mismo principio que usan los grandes modelos de lenguaje como ChatGPT: **predecir la siguiente palabra**. La diferencia es que ellos usan redes neuronales gigantes y nosotros usaremos algo mucho mas sencillo y facil de entender: **bigramas**.

## Recorrido del proyecto

1. **Preparar el corpus** - limpiar el texto y separarlo en oraciones.
2. **Construir el vocabulario** - asignar un numero a cada palabra.
3. **Contar bigramas** - que pares de palabras aparecen juntos.
4. **Calcular probabilidades** - que probabilidad hay de que tras `w1` venga `w2`.
5. **Muestrear** - elegir la siguiente palabra respetando esas probabilidades.
6. **Generar frases** - encadenar palabras para formar una oracion completa.
7. **Ida y vuelta** - codificar, decodificar y comprobar que no se pierde informacion.
8. **Guardar y cargar el modelo** - para no tener que reentrenar cada vez.
9. **Probar todo** con pruebas unitarias.

## Que necesitas saber antes de empezar

- Variables, listas y diccionarios en Python.
- Bucles `for` y condicionales `if`.
- Funciones con `def` y `return`.

No hace falta saber nada de inteligencia artificial. Vamos a construirla desde cero.

---
# Concepto clave: que es un bigrama

Un **bigrama** es simplemente **un par de palabras consecutivas** dentro de una oracion.

## Ejemplo

Tomemos la oracion:

> *"la inteligencia artificial es el futuro"*

Sus bigramas son todos los pares consecutivos:

| Bigrama | Se lee como |
|---|---|
| `('la', 'inteligencia')` | despues de *la* viene *inteligencia* |
| `('inteligencia', 'artificial')` | despues de *inteligencia* viene *artificial* |
| `('artificial', 'es')` | despues de *artificial* viene *es* |
| `('es', 'el')` | despues de *es* viene *el* |
| `('el', 'futuro')` | despues de *el* viene *futuro* |

Una oracion de **N** palabras tiene **N - 1** bigramas.

## Por que nos interesan los bigramas

Porque capturan **como se encadenan las palabras en un idioma**. Si contamos muchos bigramas en un corpus grande, podemos aprender cosas como:

- Despues de `'el'` suele venir `'futuro'`, `'problema'`, `'mundo'`...
- Despues de `'la'` suele venir `'inteligencia'`, `'tecnologia'`, `'vida'`...
- Despues de `'inteligencia'` casi siempre viene `'artificial'`.

Con esas estadisticas, podemos **generar** texto nuevo: empezamos por una palabra y vamos eligiendo la siguiente segun lo que hayamos aprendido.

---
# Concepto clave: que es el contexto

En un modelo de bigramas, el **contexto** de una palabra es simplemente **la palabra que la precede**.

## Ejemplo

En la oracion `'la inteligencia artificial'`:

- El contexto de `'inteligencia'` es `'la'`.
- El contexto de `'artificial'` es `'inteligencia'`.
- El contexto de `'la'` es **nada** (es la primera palabra, no tiene contexto).

Cuando contamos bigramas, tambien contamos **cuantas veces aparece cada palabra como contexto**. Ese numero se llama **frecuencia del contexto** y nos servira como **denominador** al calcular probabilidades.

## Ejemplo con numeros

Supongamos que en el corpus aparece:

- `'el'` como contexto **10 veces**.
- El bigrama `('el', 'futuro')` **3 veces**.
- El bigrama `('el', 'problema')` **1 vez**.

Eso significa que, de las 10 veces que aparecio `'el'` como inicio de un bigrama, en 3 ocasiones le siguio `'futuro'`, en 1 le siguio `'problema'`, y las 6 restantes le siguieron otras palabras.

---
# Concepto clave: como se calcula la probabilidad

La pregunta que queremos responder es:

> *Dada la palabra `w1`, cual es la probabilidad de que la siguiente palabra sea `w2`?*

Se escribe `P(w2 | w1)` y significa *"probabilidad de `w2` dado `w1`"*.

## Formula basica

```
                    veces que aparece (w1, w2)
P(w2 | w1) = -------------------------------------
                    veces que aparece w1 como contexto
```

Es decir: **casos favorables dividido por casos totales**.

## Ejemplo

Con los numeros anteriores:

- `('el', 'futuro')` aparece **3 veces**.
- `'el'` como contexto aparece **10 veces**.

```
P('futuro' | 'el') = 3 / 10 = 0.30   -> 30% de probabilidad
P('problema' | 'el') = 1 / 10 = 0.10 -> 10% de probabilidad
```

## El problema de la probabilidad cero

Y si un bigrama **nunca** aparecio? Por ejemplo `('el', 'pinguino')`. Su probabilidad seria:

```
P('pinguino' | 'el') = 0 / 10 = 0
```

Eso es un problema, porque significa que **el modelo jamas generara esa transicion**, aunque sea perfectamente valida en espanol. Y si multiplicamos muchas probabilidades y una es 0, el resultado es 0: la frase entera se anula.

## Solucion: suavizado de Laplace (add-one)

La idea es **sumar 1 a todos los conteos**, para que ninguna transicion tenga probabilidad 0. Y sumar el tamano del vocabulario (`V`) al denominador, para que el total siga sumando 1:

```
                    bigramas[(w1, w2)] + 1
P(w2 | w1) = ----------------------------------
                    contextos[w1] + V
```

Donde **V** es el numero de palabras distintas del vocabulario.

Con esto:

- Un bigrama visto 3 veces aporta un poco mas.
- Un bigrama nunca visto aporta un poquito (no cero).
- La suma de todas las probabilidades sigue siendo 1.

Es como repartir un poquito de probabilidad a todas las palabras posibles, quitandosela proporcionalmente a las que ya tenian mucho.

## Nota importante sobre `<UNK>`

`<UNK>` **forma parte del vocabulario** y **tambien recibe probabilidad** con Laplace. Eso es lo correcto: matematicamente, la distribucion tiene que sumar 1 sobre **todas** las palabras del vocabulario, incluida `<UNK>`.

Mas adelante, cuando **generemos** texto (Etapa 4), filtraremos `<UNK>` de la lista de candidatas para no escribir literalmente esa etiqueta en una frase. Pero en la **distribucion de probabilidad** si debe aparecer, porque representa la probabilidad de que "aqui venga una palabra desconocida".

---
# Caja de herramientas: funciones base

Antes de construir el modelo, cargamos las funciones del tokenizador que ya conocemos. Las usaremos en todas las etapas:

- `limpiar_texto` -> minusculas y sin puntuacion.
- `tokenizar` -> dividir en palabras.
- `construir_vocabulario_codificar` -> palabra a ID.
- `construir_vocabulario_decodificar` -> ID a palabra.
- `codificar_frase` / `decodificar_secuencia` -> conversion entre texto y numeros.

Ejecuta la siguiente celda tal cual: solo carga funciones, no hace nada mas.

In [ ]:
def limpiar_texto(texto, caracteres_a_quitar=None):
    """Pasa a minusculas y elimina puntuacion basica."""
    if caracteres_a_quitar is None:
        caracteres_a_quitar = ['.', ',', '!', '¿', '?', '¡', '(', ')', '"']
    texto_limpio = texto.lower()
    for caracter in caracteres_a_quitar:
        texto_limpio = texto_limpio.replace(caracter, '')
    return texto_limpio


def tokenizar(texto_limpio):
    """Divide el texto limpio en una lista de palabras."""
    return texto_limpio.split()


def construir_vocabulario_codificar(palabras, token_unk='<UNK>'):
    """Crea el diccionario palabra -> ID. El ID 0 es para <UNK>."""
    vocab = {token_unk: 0}
    for palabra in palabras:
        if palabra not in vocab:
            vocab[palabra] = len(vocab)
    return vocab


def construir_vocabulario_decodificar(vocab_codificar):
    """Invierte el vocabulario para obtener ID -> palabra."""
    return {id_num: palabra for palabra, id_num in vocab_codificar.items()}


def codificar_frase(frase_nueva, vocab_codificar):
    """Convierte una frase en IDs. Palabras desconocidas -> 0 (<UNK>)."""
    palabras = tokenizar(limpiar_texto(frase_nueva))
    return [vocab_codificar.get(p, 0) for p in palabras]


def decodificar_id(codigo, vocab_decodificar):
    """Devuelve la palabra de un ID o <UNK> si no existe."""
    return vocab_decodificar.get(codigo, '<UNK>')


def decodificar_secuencia(secuencia_numerica, vocab_decodificar):
    """Convierte una lista de IDs en una cadena de texto."""
    palabras = [decodificar_id(c, vocab_decodificar) for c in secuencia_numerica]
    return ' '.join(palabras)


def mostrar_vocab_aprendido(vocab):
    """Imprime el vocabulario de forma legible."""
    for clave, valor in vocab.items():
        print(f"{clave} : {valor}")


print("Funciones base cargadas correctamente.")

Funciones base cargadas correctamente.


---
# Etapa 1 - Preparar el corpus y el vocabulario

## Que queremos conseguir en esta etapa

Tener el texto listo para analizarlo:

1. **Limpiar** cada oracion (minusculas, sin puntuacion).
2. **Separar** el corpus en oraciones individuales.
3. **Construir** un vocabulario: cada palabra unica recibe un ID.

## Por que guardamos las oraciones por separado

Porque si unimos todo el corpus en una sola lista de palabras, el **ultimo token de una oracion** quedaria emparejado con el **primero de la siguiente**, creando bigramas **falsos**.

Ejemplo: si unimos `"el futuro"` con `"los diccionarios"`, apareceria el bigrama `('futuro', 'los')`, que nunca ocurre en espanol.

**Regla de oro:** los bigramas solo existen **dentro** de los limites de cada oracion.

## Las funciones que vamos a escribir

- `preparar_corpus(texto_crudo)` -> devuelve una **lista de oraciones**, cada una como lista de tokens.
- `recolectar_vocabulario(oraciones)` -> recorre todas las oraciones y construye el vocabulario reutilizando `construir_vocabulario_codificar`.

In [ ]:
def preparar_corpus(texto_crudo):
    """Devuelve una lista de oraciones, cada una como lista de tokens limpios."""
    oraciones = []
    for linea in texto_crudo.strip().split('\n'):
        limpia = limpiar_texto(linea)
        tokens = tokenizar(limpia)
        if tokens:                      # ignoramos lineas vacias
            oraciones.append(tokens)
    return oraciones


def recolectar_vocabulario(oraciones):
    """Construye el vocabulario (palabra -> ID) a partir de una lista de oraciones."""
    todas_las_palabras = [p for oracion in oraciones for p in oracion]
    return construir_vocabulario_codificar(todas_las_palabras)


# --- Prueba rapida ---
corpus_demo = """La Inteligencia Artificial es el futuro de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante."""

oraciones = preparar_corpus(corpus_demo)
vocab_cod = recolectar_vocabulario(oraciones)
vocab_dec = construir_vocabulario_decodificar(vocab_cod)

print("Numero de oraciones:", len(oraciones))
for i, o in enumerate(oraciones, 1):
    print(f"   Oracion {i}: {o}")

print("\nTamano del vocabulario:", len(vocab_cod))
print("Palabras del vocabulario:")
mostrar_vocab_aprendido(vocab_cod)

Numero de oraciones: 3
   Oracion 1: ['la', 'inteligencia', 'artificial', 'es', 'el', 'futuro', 'de', 'la', 'tecnologia']
   Oracion 2: ['los', 'diccionarios', 'en', 'python', 'son', 'extremadamente', 'rapidos']
   Oracion 3: ['las', 'tablas', 'hash', 'permiten', 'busquedas', 'en', 'tiempo', 'constante']

Tamano del vocabulario: 23
Palabras del vocabulario:
<UNK> : 0
la : 1
inteligencia : 2
artificial : 3
es : 4
el : 5
futuro : 6
de : 7
tecnologia : 8
los : 9
diccionarios : 10
en : 11
python : 12
son : 13
extremadamente : 14
rapidos : 15
las : 16
tablas : 17
hash : 18
permiten : 19
busquedas : 20
tiempo : 21
constante : 22


## Preguntas de control (Etapa 1)

1. Por que guardamos las oraciones por separado en lugar de tokenizar todo el corpus de golpe?
2. Cual es el ID de `<UNK>`? Por que crees que reservamos ese numero?
3. Que funcion de las que ya conociamos hemos reutilizado para construir el vocabulario?

> Respuestas rapidas:
> 1. Para no crear bigramas falsos entre el final de una oracion y el inicio de la siguiente.
> 2. Es 0. Se reserva para cualquier palabra que el modelo no conozca.
> 3. `construir_vocabulario_codificar`.

---
# Etapa 2 - Contar bigramas y contextos

## Que queremos conseguir

Recorrer todas las oraciones y llenar dos diccionarios:

### 1. `bigramas`
Cuantas veces aparece cada par `(w1, w2)`.

```
{
  ('la', 'inteligencia'): 1,
  ('el', 'futuro'): 1,
  ('en', 'python'): 1,
  ...
}
```

### 2. `contextos`
Cuantas veces aparece cada palabra **como primera de un bigrama**.

```
{
  'la': 3,
  'el': 2,
  'en': 1,
  ...
}
```

Este segundo diccionario sera el **denominador** al calcular probabilidades.

## Como lo hacemos sin `Counter`

Con un diccionario normal y el metodo `.get(clave, 0)`:

```python
bigramas[par] = bigramas.get(par, 0) + 1
```

Esto significa: *"si ya existia, sumale 1; si no existia, empieza desde 0 y anade 1"*.

Es exactamente lo que hace `Counter` por dentro, pero escrito a mano, para que veas la mecanica.

In [ ]:
def contar_bigramas(oraciones):
    """Devuelve un dict con los pares (w1, w2) y su frecuencia."""
    bigramas = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 1):
            par = (tokens[i], tokens[i + 1])
            bigramas[par] = bigramas.get(par, 0) + 1
    return bigramas


def contar_contextos(oraciones):
    """Cuenta cuantas veces aparece cada palabra como primera de un bigrama."""
    contextos = {}
    for tokens in oraciones:
        for i in range(len(tokens) - 1):
            palabra = tokens[i]
            contextos[palabra] = contextos.get(palabra, 0) + 1
    return contextos


def top_n(diccionario, n):
    """Devuelve los n pares (clave, valor) mas frecuentes."""
    return sorted(diccionario.items(), key=lambda kv: kv[1], reverse=True)[:n]


# --- Prueba rapida ---
bigramas = contar_bigramas(oraciones)
contextos = contar_contextos(oraciones)

print("Bigramas mas frecuentes:")
for par, freq in top_n(bigramas, 5):
    print(f"   {par} -> {freq}")

print("\nContextos mas frecuentes (denominadores de la probabilidad):")
for palabra, freq in top_n(contextos, 5):
    print(f"   {palabra} -> {freq}")

print("\nComprobacion manual:")
print(f"   El bigrama ('el', 'futuro') aparece {bigramas.get(('el','futuro'), 0)} vez/veces")
print(f"   La palabra 'el' es contexto {contextos.get('el', 0)} vez/veces")
print(f"   Por tanto P('futuro' | 'el') = {bigramas.get(('el','futuro'),0)} / {contextos.get('el',0)}")

Bigramas mas frecuentes:
   ('la', 'inteligencia') -> 1
   ('inteligencia', 'artificial') -> 1
   ('artificial', 'es') -> 1
   ('es', 'el') -> 1
   ('el', 'futuro') -> 1

Contextos mas frecuentes (denominadores de la probabilidad):
   la -> 2
   en -> 2
   inteligencia -> 1
   artificial -> 1
   es -> 1

Comprobacion manual:
   El bigrama ('el', 'futuro') aparece 1 vez/veces
   La palabra 'el' es contexto 1 vez/veces
   Por tanto P('futuro' | 'el') = 1 / 1


## Preguntas de control (Etapa 2)

1. Cual es la diferencia entre `bigramas` y `contextos`?
2. Si `contextos['la'] = 3`, que significa ese 3?
3. Por que usamos `.get(par, 0)` en lugar de `bigramas[par]` directamente?

> Respuestas:
> 1. `bigramas` cuenta pares de palabras; `contextos` cuenta cuantas veces cada palabra inicia un bigrama.
> 2. Que `'la'` aparece 3 veces como primera palabra de un bigrama.
> 3. Porque `.get` devuelve 0 si la clave no existe, mientras que `bigramas[par]` lanzaria `KeyError`.

---
# Etapa 3 - Probabilidades con suavizado de Laplace

## La formula

Recordemos:

```
                    bigramas[(w1, w2)] + 1
P(w2 | w1) = ----------------------------------
                    contextos[w1] + V
```

- `+1` en el numerador -> suavizado (nada tiene probabilidad 0).
- `+V` en el denominador -> compensa los +1 y mantiene la suma en 1.
- **V** = numero de palabras del vocabulario.

## Ejemplo numerico

Supongamos:

- `bigramas[('el', 'futuro')] = 3`
- `contextos['el'] = 10`
- `V = 50`

Entonces:

```
P('futuro' | 'el') = (3 + 1) / (10 + 50) = 4 / 60 = 0.0667
```

Y para una palabra nunca vista:

```
P('pinguino' | 'el') = (0 + 1) / (10 + 50) = 1 / 60 = 0.0167
```

Pequenito, pero **mayor que cero**. Eso es exactamente lo que queremos.

## Nota importante sobre `<UNK>`

`<UNK>` **forma parte del vocabulario** y **tambien recibe probabilidad** con Laplace. La distribucion debe sumar 1 sobre **todas** las palabras del vocabulario, incluida `<UNK>`.

En la Etapa 4, al muestrear, filtraremos `<UNK>` para no escribirlo en una frase generada. Pero en la distribucion de probabilidad si aparece.

## Las funciones que vamos a escribir

- `probabilidad_bigrama(w1, w2, ...)` -> devuelve un numero entre 0 y 1.
- `distribucion_siguiente(w1, ...)` -> devuelve un diccionario `{palabra: probabilidad}` con **todas** las palabras del vocabulario (incluida `<UNK>`) como posibles continuaciones. La suma de todas las probabilidades es exactamente 1.

In [ ]:
def probabilidad_bigrama(w1, w2, bigramas, contextos, vocab_size):
    """P(w2 | w1) con suavizado de Laplace."""
    conteo_par = bigramas.get((w1, w2), 0)
    conteo_contexto = contextos.get(w1, 0)
    return (conteo_par + 1) / (conteo_contexto + vocab_size)


def distribucion_siguiente(w1, bigramas, contextos, vocab_decodificar):
    """Devuelve un dict {palabra: probabilidad} de TODAS las palabras del vocabulario."""
    vocab_size = len(vocab_decodificar)
    distribucion = {}
    for palabra in vocab_decodificar.values():
        distribucion[palabra] = probabilidad_bigrama(w1, palabra, bigramas, contextos, vocab_size)
    return distribucion


# --- Prueba rapida ---
V = len(vocab_cod)
print("Tamano del vocabulario V =", V)

print("\nProbabilidades de algunas continuaciones de 'el':")
for w in ['futuro', 'python', 'tecnologia', 'hash']:
    p = probabilidad_bigrama('el', w, bigramas, contextos, V)
    print(f"   P('{w}' | 'el') = {p:.4f}")

dist = distribucion_siguiente('el', bigramas, contextos, vocab_dec)
print("\nTop 5 continuaciones de 'el':")
for palabra, p in top_n(dist, 5):
    print(f"   {palabra:15s} {p:.4f}")

suma = sum(dist.values())
print(f"\nSuma de todas las probabilidades: {suma:.6f}  (debe ser = 1)")
print(f"La distribucion contiene <UNK>? {'<UNK>' in dist}")

Tamano del vocabulario V = 23

Probabilidades de algunas continuaciones de 'el':
   P('futuro' | 'el') = 0.0833
   P('python' | 'el') = 0.0417
   P('tecnologia' | 'el') = 0.0417
   P('hash' | 'el') = 0.0417

Top 5 continuaciones de 'el':
   futuro          0.0833
   <UNK>           0.0417
   la              0.0417
   inteligencia    0.0417
   artificial      0.0417

Suma de todas las probabilidades: 1.000000  (debe ser = 1)
La distribucion contiene <UNK>? True


## Preguntas de control (Etapa 3)

1. Por que sumamos 1 al numerador?
2. Por que sumamos `V` al denominador?
3. Si `V` fuera 100 y `contextos['el'] = 5`, cual seria la probabilidad de una palabra nunca vista tras `'el'`?
4. Por que la distribucion incluye `<UNK>` y no lo excluimos?

> Respuestas:
> 1. Para que ninguna transicion tenga probabilidad 0.
> 2. Para compensar los +1 y mantener la suma de probabilidades igual a 1.
> 3. `1 / (5 + 100) = 1/105 = 0.0095`.
> 4. Porque `<UNK>` es una palabra mas del vocabulario y la distribucion debe sumar 1 sobre todas las palabras. Si lo excluyeramos, la suma no daria 1.

---
# Etapa 4 - Muestrear la siguiente palabra

## Que es muestrear

**Muestrear** = elegir un elemento al azar **respetando sus probabilidades**.

Si tenemos:

- `'futuro'` con probabilidad 0.30
- `'problema'` con probabilidad 0.10
- `'pinguino'` con probabilidad 0.01

y muestreamos muchas veces, `'futuro'` debe salir ~30% de las veces, `'problema'` ~10%, y `'pinguino'` ~1%.

## La herramienta: `random.choices`

Python tiene una funcion pensada exactamente para esto:

```python
random.choices(palabras, weights=probabilidades, k=1)[0]
```

- `palabras`: la lista de candidatos.
- `weights`: sus pesos (las probabilidades).
- `k=1`: cuantos queremos sacar (uno).
- Devuelve una lista, por eso `[0]` para quedarnos con el primer (y unico) elemento.

## Y `<UNK>`?

La distribucion que calculamos en la Etapa 3 incluye `<UNK>` (para que sume 1 y sea matematicamente correcta). Pero **no queremos escribir `<UNK>` en una frase generada**: quedaria feo y confuso.

Solucion: al muestrear, **filtramos `<UNK>` de la lista de candidatas**. Es decir, usamos la distribucion completa para el calculo, pero solo elegimos entre las palabras "reales".

`random.choices` normaliza internamente los pesos, asi que no hace falta renormalizar a mano despues de filtrar.

## Por que fijamos `random.seed(...)`

Sin semilla, cada ejecucion daria resultados distintos. Con semilla, obtenemos **siempre lo mismo**, lo cual es imprescindible para que las pruebas unitarias sean fiables y reproducibles.

In [ ]:
import random

def muestrear_siguiente_palabra(w1, bigramas, contextos, vocab_decodificar):
    """Dada una palabra, devuelve la siguiente muestreada segun P(· | w1),
    excluyendo <UNK> de las candidatas."""
    distribucion = distribucion_siguiente(w1, bigramas, contextos, vocab_decodificar)
    # Filtramos <UNK>: no queremos que aparezca en texto generado
    candidatas = {p: pr for p, pr in distribucion.items() if p != '<UNK>'}
    palabras = list(candidatas.keys())
    pesos    = list(candidatas.values())
    return random.choices(palabras, weights=pesos, k=1)[0]


# --- Prueba rapida ---
random.seed(42)
print("10 muestras de P(· | 'el'):")
for _ in range(10):
    print("   ", muestrear_siguiente_palabra('el', bigramas, contextos, vocab_dec))

# Comprobacion estadistica: coincide con lo teorico?
random.seed(42)
N = 5000
conteos = {}
for _ in range(N):
    w = muestrear_siguiente_palabra('el', bigramas, contextos, vocab_dec)
    conteos[w] = conteos.get(w, 0) + 1

print(f"\nFrecuencia empirica vs teorica ({N} muestras):")
print(f"   {'palabra':15s} {'empirica':>10s} {'teorica':>10s}")
for palabra, freq in top_n(conteos, 5):
    empirica = freq / N
    teorica = probabilidad_bigrama('el', palabra, bigramas, contextos, V)
    print(f"   {palabra:15s} {empirica:10.4f} {teorica:10.4f}")

print("\nCuantas mas muestras, mas se parecen empirica y teorica (ley de los grandes numeros).")

10 muestras de P(· | 'el'):
    extremadamente
    la
    futuro
    futuro
    las
    rapidos
    busquedas
    inteligencia
    los
    la

Frecuencia empirica vs teorica (5000 muestras):
   palabra           empirica    teorica
   futuro              0.0912     0.0833
   python              0.0486     0.0417
   permiten            0.0474     0.0417
   rapidos             0.0472     0.0417
   extremadamente      0.0462     0.0417

Cuantas mas muestras, mas se parecen empirica y teorica (ley de los grandes numeros).


## Preguntas de control (Etapa 4)

1. Que diferencia hay entre **elegir la mas probable** y **muestrear**?
2. Por que no pasamos los conteos brutos como pesos en lugar de las probabilidades?
3. Para que sirve `random.seed`?
4. Por que la distribucion incluye `<UNK>` pero el muestreo no lo devuelve?

> Respuestas:
> 1. Elegir la mas probable siempre da la misma frase (determinista). Muestrear da variedad, respetando las probabilidades.
> 2. `random.choices` normaliza los pesos internamente, asi que tecnicamente funcionaria; pero pasar probabilidades ya normalizadas es mas claro y coherente con la teoria.
> 3. Para que los resultados sean reproducibles.
> 4. La distribucion debe sumar 1, y para ello incluye `<UNK>`. Pero al generar texto filtramos `<UNK>` para no escribir esa etiqueta tecnica en la frase.

---
# Etapa 5 - Generar una frase completa

## La idea central del proyecto

Ya sabemos **muestrear una palabra** dada la anterior. Ahora encadenamos:

```
semilla -> muestrear -> muestrear -> muestrear -> ...
```

Cada palabra que obtenemos pasa a ser la nueva "actual", y muestreamos la siguiente. Repetimos hasta tener N palabras.

Esto es **exactamente** lo que hacen los modelos de lenguaje: predecir la siguiente palabra y realimentarla. La diferencia es que ellos miran toda la frase previa; nosotros solo miramos **la ultima palabra** (por eso es un modelo de **bigramas**).

## Criterios de parada

Para este ejercicio paramos con un **numero fijo** de palabras (`n_palabras`). Es lo mas simple y reproducible.

## Por que las frases salen raras

Porque el modelo solo tiene **memoria de una palabra**. No sabe que dijo dos pasos atras. Por eso puede generar cosas como *"el futuro de la tecnologia son rapidos"*, que es agramatical.

Los **trigramas** (mirar dos palabras atras) o las **redes neuronales** (mirar toda la frase) resuelven esto. Pero el bigrama ya captura la esencia del problema y es la base conceptual de todo lo demas.

In [ ]:
def generar_frase(semilla, bigramas, contextos, vocab_decodificar, n_palabras=10):
    """Genera una frase de n_palabras a partir de una semilla."""
    frase = [semilla]
    actual = semilla
    for _ in range(n_palabras - 1):
        actual = muestrear_siguiente_palabra(actual, bigramas, contextos, vocab_decodificar)
        frase.append(actual)
    return frase


# --- Prueba rapida ---
random.seed(42)
print("Tres frases generadas desde 'el':")
for i in range(3):
    frase = generar_frase('el', bigramas, contextos, vocab_dec, n_palabras=10)
    print(f"   Frase {i+1}: {' '.join(frase)}")

print("\nFrases desde distintas semillas:")
random.seed(1)
for s in ['la', 'los', 'el', 'python', 'hash', 'inteligencia']:
    frase = generar_frase(s, bigramas, contextos, vocab_dec, n_palabras=8)
    print(f"   [{s:15s}] {' '.join(frase)}")

Tres frases generadas desde 'el':
   Frase 1: el extremadamente la futuro futuro las las busquedas inteligencia los
   Frase 2: el la el en la es extremadamente son futuro son
   Frase 3: el hash la hash tablas tecnologia es constante tecnologia artificial

Frases desde distintas semillas:
   [la             ] la artificial permiten hash futuro en en rapidos
   [los            ] los hash artificial la permiten diccionarios tablas la
   [el             ] el diccionarios las futuro tiempo tiempo la la
   [python         ] python son tiempo los el los la el
   [hash           ] hash en python futuro futuro futuro diccionarios de
   [inteligencia   ] inteligencia la permiten son extremadamente el constante permiten


## Preguntas de control (Etapa 5)

1. Por que la frase generada puede ser distinta cada vez aunque usemos la misma semilla?
2. Que limitacion fundamental tiene un modelo de bigramas respecto a un modelo mas avanzado?
3. Que pasaria si la semilla no esta en el vocabulario?

> Respuestas:
> 1. Si **no** fijamos la semilla, si cambia. Si **si** la fijamos (`random.seed`), la frase es siempre la misma. Prueba a ejecutar la celda varias veces sin cambiar la semilla y veras que repite.
> 2. Solo mira **una** palabra atras. No tiene memoria de la frase completa.
> 3. Como `contextos.get(semilla, 0)` devuelve 0, se aplica Laplace sobre 0 y todas las palabras tienen la misma probabilidad pequena. El modelo generara ruido. Es robusto (no falla), pero sin sentido.

---
# Etapa 6 - Ida y vuelta: codificar, decodificar, comparar

## Por que hacemos esto

Hasta ahora teniamos dos mundos separados:

- El **tokenizador** (palabras y numeros).
- El **modelo de bigramas** (genera texto).

Aqui los unimos para comprobar que **no se pierde informacion** en el camino:

```
tokens  ->  codificar  ->  IDs  ->  decodificar  ->  tokens
```

Como la frase la genero el propio modelo, **todas** sus palabras existen en el vocabulario. Por tanto la ida y vuelta debe ser **identica**.

## El caso interesante: palabra desconocida

Si metemos a proposito una palabra que **no esta** en el vocabulario:

- En modo estricto se convierte en `0` -> `<UNK>` al decodificar.
- El sistema **no colapsa**: sigue funcionando.

Eso demuestra la **robustez** del tokenizador.

In [ ]:
def ida_y_vuelta(frase_tokens, vocab_cod, vocab_dec):
    """Codifica una lista de tokens y la vuelve a decodificar."""
    ids = [vocab_cod.get(p, 0) for p in frase_tokens]
    reconstruida = decodificar_secuencia(ids, vocab_dec)
    return ids, reconstruida.split()


def comparar_frases(original, reconstruida):
    """Devuelve un dict con el resultado de la comparacion."""
    return {
        "original": original,
        "reconstruida": reconstruida,
        "longitudes_iguales": len(original) == len(reconstruida),
        "identicas": original == reconstruida,
        "diferencias": [
            (i, a, b)
            for i, (a, b) in enumerate(zip(original, reconstruida))
            if a != b
        ],
    }


# --- Prueba 1: frase generada por el modelo ---
random.seed(42)
frase = generar_frase('el', bigramas, contextos, vocab_dec, n_palabras=10)
ids, reconstruida = ida_y_vuelta(frase, vocab_cod, vocab_dec)
resultado = comparar_frases(frase, reconstruida)

print("Frase generada   :", " ".join(frase))
print("IDs              :", ids)
print("Reconstruida     :", " ".join(reconstruida))
print("Identicas?       ", resultado["identicas"])
print("Diferencias      ", resultado["diferencias"])

# --- Prueba 2: frase con palabra desconocida a proposito ---
print("\n--- Prueba con palabra desconocida ---")
frase_desconocida = ['el', 'futuro', 'cuantico', 'de', 'la', 'ia']
ids2, reconstruida2 = ida_y_vuelta(frase_desconocida, vocab_cod, vocab_dec)
resultado2 = comparar_frases(frase_desconocida, reconstruida2)

print("Original     :", " ".join(frase_desconocida))
print("IDs          :", ids2)
print("Reconstruida :", " ".join(reconstruida2))
print("Identicas?   ", resultado2["identicas"])
print("Diferencias  ", resultado2["diferencias"])

Frase generada   : el extremadamente la futuro futuro las las busquedas inteligencia los
IDs              : [5, 14, 1, 6, 6, 16, 16, 20, 2, 9]
Reconstruida     : el extremadamente la futuro futuro las las busquedas inteligencia los
Identicas?        True
Diferencias       []

--- Prueba con palabra desconocida ---
Original     : el futuro cuantico de la ia
IDs          : [5, 6, 0, 7, 1, 0]
Reconstruida : el futuro <UNK> de la <UNK>
Identicas?    False
Diferencias   [(2, 'cuantico', '<UNK>'), (5, 'ia', '<UNK>')]


## Preguntas de control (Etapa 6)

1. Por que la ida y vuelta es exacta para frases generadas por el modelo?
2. Que ID recibe una palabra desconocida en modo estricto?
3. Que aparece al decodificar ese ID?

> Respuestas:
> 1. Porque todas las palabras generadas salen del vocabulario, asi que su codificacion y decodificacion son biunivocas.
> 2. El ID 0 (reservado para `<UNK>`).
> 3. La etiqueta `<UNK>`.

---
# Etapa 7 - Guardar y cargar el modelo en JSON

## Por que guardar el modelo

Entrenar es rapido con nuestro corpus pequeno, pero con un corpus grande tardaria mucho. Guardar el modelo nos permite:

- **Reutilizarlo** sin reentrenar.
- **Compartirlo** con otros.
- **Inspeccionarlo** a mano (es un JSON legible).

## Que guardamos

Cuatro cosas:

1. **Version** del formato (por si lo cambiamos en el futuro).
2. **`vocab_cod`** (palabra a ID).
3. **`bigramas`** (pares a conteo).
4. **`contextos`** (palabra a conteo).

**No** guardamos `vocab_dec` porque se reconstruye invirtiendo `vocab_cod`. Asi evitamos inconsistencias.

## El problema de las tuplas como claves

JSON **solo admite claves de tipo texto**. Pero nuestras claves de `bigramas` son **tuplas**: `('el', 'futuro')`. No se pueden guardar directamente.

**Solucion**: convertir cada tupla en un texto usando un separador:

```
('el', 'futuro')  ->  "el ||| futuro"
```

Al cargar, deshacemos la conversion con `.split(" ||| ")`.

Elegimos `" ||| "` como separador porque:
- Es **legible** al abrir el JSON.
- Es **muy poco probable** que aparezca dentro de una palabra real.
- Se ve claramente como separador.

In [ ]:
import json

SEPARADOR = " ||| "


def bigramas_a_json(bigramas):
    """Convierte dict {(w1, w2): n}  ->  dict {'w1 ||| w2': n}."""
    return {f"{w1}{SEPARADOR}{w2}": n for (w1, w2), n in bigramas.items()}


def json_a_bigramas(bigramas_json):
    """Convierte dict {'w1 ||| w2': n}  ->  dict {(w1, w2): n}."""
    return {tuple(clave.split(SEPARADOR)): n for clave, n in bigramas_json.items()}


def guardar_modelo(ruta, vocab_cod, bigramas, contextos):
    """Guarda el modelo completo en un archivo JSON."""
    modelo = {
        "version": 1,
        "vocab_cod": vocab_cod,
        "bigramas": bigramas_a_json(bigramas),
        "contextos": contextos,
    }
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(modelo, f, ensure_ascii=False, indent=2)


def cargar_modelo(ruta):
    """Carga el modelo desde un archivo JSON y reconstruye vocab_dec."""
    with open(ruta, "r", encoding="utf-8") as f:
        modelo = json.load(f)
    vocab_cod = modelo["vocab_cod"]
    bigramas = json_a_bigramas(modelo["bigramas"])
    contextos = modelo["contextos"]
    vocab_dec = construir_vocabulario_decodificar(vocab_cod)
    return vocab_cod, vocab_dec, bigramas, contextos


# --- Prueba de ida y vuelta ---
guardar_modelo("modelo_bigramas.json", vocab_cod, bigramas, contextos)
vc2, vd2, bg2, cx2 = cargar_modelo("modelo_bigramas.json")

print("Vocabulario igual: ", vocab_cod == vc2)
print("Bigramas iguales:  ", bigramas  == bg2)
print("Contextos iguales: ", contextos == cx2)
print("Tamano vocab:      ", len(vc2))
print("N bigramas:        ", len(bg2))

# Generamos con el modelo cargado: deberia dar la misma frase
random.seed(42)
frase_cargada = generar_frase('el', bg2, cx2, vd2, n_palabras=10)
print("\nFrase con el modelo cargado:")
print("   ", " ".join(frase_cargada))

# Inspeccion del JSON
print("\n--- Primeras lineas del JSON ---")
with open("modelo_bigramas.json", "r", encoding="utf-8") as f:
    print(f.read()[:600])

Vocabulario igual:  True
Bigramas iguales:   True
Contextos iguales:  True
Tamano vocab:       23
N bigramas:         21

Frase con el modelo cargado:
    el extremadamente la futuro futuro las las busquedas inteligencia los

--- Primeras lineas del JSON ---
{
  "version": 1,
  "vocab_cod": {
    "<UNK>": 0,
    "la": 1,
    "inteligencia": 2,
    "artificial": 3,
    "es": 4,
    "el": 5,
    "futuro": 6,
    "de": 7,
    "tecnologia": 8,
    "los": 9,
    "diccionarios": 10,
    "en": 11,
    "python": 12,
    "son": 13,
    "extremadamente": 14,
    "rapidos": 15,
    "las": 16,
    "tablas": 17,
    "hash": 18,
    "permiten": 19,
    "busquedas": 20,
    "tiempo": 21,
    "constante": 22
  },
  "bigramas": {
    "la ||| inteligencia": 1,
    "inteligencia ||| artificial": 1,
    "artificial ||| es": 1,
    "es ||| el": 1,
    "el ||| futuro":


## Preguntas de control (Etapa 7)

1. Por que no guardamos `vocab_dec` en el JSON?
2. Por que las claves de `bigramas` no se pueden guardar tal cual?
3. Por que la frase generada con el modelo cargado es **identica** a la del modelo original?

> Respuestas:
> 1. Porque se reconstruye invirtiendo `vocab_cod`. Ahorra espacio y evita inconsistencias.
> 2. Porque JSON solo admite claves de tipo texto, no tuplas.
> 3. Porque los conteos son exactamente los mismos y fijamos la misma semilla.

---
# Etapa 8 - Pruebas unitarias

## Que es una prueba unitaria

Es un trozo de codigo que **comprueba automaticamente** que una funcion hace lo que debe. Si algun dia cambias algo y rompes una funcion, las pruebas te avisan de inmediato.

## La herramienta: `assert`

En Python, la instruccion `assert condicion, mensaje` comprueba que la condicion es cierta. Si **no** lo es, lanza un `AssertionError` con el mensaje. Es la forma mas sencilla de escribir pruebas.

## Que vamos a probar

Al menos una prueba por etapa:

| Etapa | Que comprobamos |
|---|---|
| 1 | El corpus se separa bien y el vocabulario tiene `<UNK>=0` |
| 2 | Los bigramas no cruzan oraciones y se cuentan bien |
| 3 | Las probabilidades suman 1 y nunca son 0 |
| 4 | Las palabras muestreadas estan en el vocabulario y nunca son `<UNK>` |
| 5 | La frase generada empieza por la semilla y tiene N palabras |
| 6 | La ida y vuelta es identica y las palabras desconocidas van a `<UNK>` |
| 7 | Guardar y cargar conserva los conteos |

In [ ]:
print("=== PRUEBAS UNITARIAS DEL MODELO DE BIGRAMAS ===\n")

# ---------- Etapa 1 ----------
print("Etapa 1: preparar corpus y vocabulario")

o = preparar_corpus("hola mundo\n\nadios mundo")
assert len(o) == 2, f"Esperaba 2 oraciones, obtuve {len(o)}"
assert o[0] == ['hola', 'mundo']

v = recolectar_vocabulario([['a', 'b', 'a'], ['b', 'c']])
assert v['<UNK>'] == 0
assert v['a'] == 1 and v['b'] == 2 and v['c'] == 3

vd = construir_vocabulario_decodificar(v)
for palabra, idn in v.items():
    assert vd[idn] == palabra

print("   OK - Todas las pruebas de la Etapa 1 pasaron\n")

# ---------- Etapa 2 ----------
print("Etapa 2: contar bigramas y contextos")

bg = contar_bigramas([['a', 'b', 'a', 'b']])
assert bg[('a', 'b')] == 2, f"Esperaba 2, obtuve {bg[('a', 'b')]}"
assert bg[('b', 'a')] == 1

# Los bigramas NO deben cruzar oraciones
bg2 = contar_bigramas([['a', 'b'], ['c', 'd']])
assert ('b', 'c') not in bg2, "Los bigramas no deben cruzar oraciones"

cx = contar_contextos([['a', 'b', 'a', 'b']])
assert cx['a'] == 2 and cx['b'] == 1

print("   OK - Todas las pruebas de la Etapa 2 pasaron\n")

# ---------- Etapa 3 ----------
print("Etapa 3: probabilidades y suavizado de Laplace")

V_prueba = len(vocab_cod)
dist = distribucion_siguiente('el', bigramas, contextos, vocab_dec)
suma = sum(dist.values())
assert abs(suma - 1.0) < 1e-6, f"La distribucion no suma 1, suma {suma}"

# La distribucion debe incluir <UNK>
assert '<UNK>' in dist, "La distribucion debe incluir <UNK>"

p_rara = probabilidad_bigrama('el', 'palabraquenoexiste', bigramas, contextos, V_prueba)
assert p_rara > 0, "Laplace debe evitar el 0"

print("   OK - Todas las pruebas de la Etapa 3 pasaron\n")

# ---------- Etapa 4 ----------
print("Etapa 4: muestreo")

random.seed(0)
for _ in range(50):
    w = muestrear_siguiente_palabra('el', bigramas, contextos, vocab_dec)
    assert w in vocab_cod, f"Palabra fuera del vocabulario: {w}"

# El muestreo nunca debe devolver <UNK>
random.seed(0)
for _ in range(200):
    w = muestrear_siguiente_palabra('el', bigramas, contextos, vocab_dec)
    assert w != '<UNK>', "El muestreo no debe devolver <UNK>"

print("   OK - Todas las pruebas de la Etapa 4 pasaron\n")

# ---------- Etapa 5 ----------
print("Etapa 5: generar frase")

random.seed(42)
fr = generar_frase('el', bigramas, contextos, vocab_dec, n_palabras=10)
assert len(fr) == 10, f"Esperaba 10 palabras, obtuve {len(fr)}"
assert fr[0] == 'el', "La frase debe empezar por la semilla"

print("   OK - Todas las pruebas de la Etapa 5 pasaron\n")

# ---------- Etapa 6 ----------
print("Etapa 6: ida y vuelta")

frase_ok = ['el', 'futuro', 'de', 'la', 'tecnologia']
ids, rec = ida_y_vuelta(frase_ok, vocab_cod, vocab_dec)
assert frase_ok == rec, f"Esperaba {frase_ok}, obtuve {rec}"

frase_mal = ['el', 'cuantico', 'futuro']
ids2, rec2 = ida_y_vuelta(frase_mal, vocab_cod, vocab_dec)
assert ids2[1] == 0, "Palabra desconocida debe ser 0"
assert rec2[1] == '<UNK>', "ID 0 debe decodificarse como <UNK>"

print("   OK - Todas las pruebas de la Etapa 6 pasaron\n")

# ---------- Etapa 7 ----------
print("Etapa 7: guardar y cargar modelo")

guardar_modelo("modelo_test.json", vocab_cod, bigramas, contextos)
vc_t, vd_t, bg_t, cx_t = cargar_modelo("modelo_test.json")
assert vc_t == vocab_cod, "Vocabulario distinto tras cargar"
assert bg_t == bigramas, "Bigramas distintos tras cargar"
assert cx_t == contextos, "Contextos distintos tras cargar"

print("   OK - Todas las pruebas de la Etapa 7 pasaron\n")

print("=== TODAS LAS PRUEBAS PASARON ===")

=== PRUEBAS UNITARIAS DEL MODELO DE BIGRAMAS ===

Etapa 1: preparar corpus y vocabulario
   OK - Todas las pruebas de la Etapa 1 pasaron

Etapa 2: contar bigramas y contextos
   OK - Todas las pruebas de la Etapa 2 pasaron

Etapa 3: probabilidades y suavizado de Laplace
   OK - Todas las pruebas de la Etapa 3 pasaron

Etapa 4: muestreo
   OK - Todas las pruebas de la Etapa 4 pasaron

Etapa 5: generar frase
   OK - Todas las pruebas de la Etapa 5 pasaron

Etapa 6: ida y vuelta
   OK - Todas las pruebas de la Etapa 6 pasaron

Etapa 7: guardar y cargar modelo
   OK - Todas las pruebas de la Etapa 7 pasaron

=== TODAS LAS PRUEBAS PASARON ===


---
# Demostracion final

Recorrido completo del sistema en cinco pasos:

1. Entrenamos el modelo con un corpus.
2. Generamos una frase desde una semilla.
3. La codificamos, la decodificamos y comprobamos que es identica.
4. Guardamos el modelo en JSON.
5. Lo cargamos desde cero y volvemos a generar la **misma** frase.

Este cierre demuestra que todas las piezas encajan y que el sistema es **robusto** y **persistente**.

In [ ]:
# 1) Entrenar
corpus_final = """La Inteligencia Artificial es el futuro de la tecnologia.
Los diccionarios en Python son extremadamente rapidos.
Las tablas hash permiten busquedas en tiempo constante.
La programacion en Python es divertida y poderosa."""

oraciones_f = preparar_corpus(corpus_final)
vocab_cod_f = recolectar_vocabulario(oraciones_f)
vocab_dec_f = construir_vocabulario_decodificar(vocab_cod_f)
bigramas_f = contar_bigramas(oraciones_f)
contextos_f = contar_contextos(oraciones_f)

print(f"Vocabulario: {len(vocab_cod_f)} palabras")
print(f"Bigramas:    {len(bigramas_f)} pares distintos")

# 2) Generar
random.seed(7)
frase_gen = generar_frase('la', bigramas_f, contextos_f, vocab_dec_f, n_palabras=12)
print("\nFrase generada:")
print("   ", " ".join(frase_gen))

# 3) Ida y vuelta
ids_gen, frase_rec = ida_y_vuelta(frase_gen, vocab_cod_f, vocab_dec_f)
print("\nIDs:", ids_gen)
print("Reconstruida:", " ".join(frase_rec))
print("Identicas?", frase_gen == frase_rec)

# 4) Guardar
guardar_modelo("modelo_final.json", vocab_cod_f, bigramas_f, contextos_f)
print("\nModelo guardado en 'modelo_final.json'")

# 5) Cargar desde cero y regenerar
vocab_cod_c, vocab_dec_c, bigramas_c, contextos_c = cargar_modelo("modelo_final.json")
random.seed(7)
frase_gen_c = generar_frase('la', bigramas_c, contextos_c, vocab_dec_c, n_palabras=12)
print("\nFrase desde el modelo cargado:")
print("   ", " ".join(frase_gen_c))
print("\nMisma frase que antes?", frase_gen == frase_gen_c)

Vocabulario: 27 palabras
Bigramas:    27 pares distintos

Frase generada:
    la tecnologia es hash inteligencia extremadamente diccionarios inteligencia son inteligencia en artificial

IDs: [1, 8, 4, 18, 2, 14, 10, 2, 13, 2, 11, 3]
Reconstruida: la tecnologia es hash inteligencia extremadamente diccionarios inteligencia son inteligencia en artificial
Identicas? True

Modelo guardado en 'modelo_final.json'

Frase desde el modelo cargado:
    la tecnologia es hash inteligencia extremadamente diccionarios inteligencia son inteligencia en artificial

Misma frase que antes? True


---
# Resumen final: que hemos aprendido

## Conceptos

- **Bigrama**: par de palabras consecutivas `(w1, w2)`.
- **Contexto**: la palabra que precede (`w1`).
- **Probabilidad de transicion**: `P(w2 | w1)` = cuantas veces aparece el par dividido por cuantas veces aparece `w1` como contexto.
- **Suavizado de Laplace**: sumar 1 al numerador y `V` al denominador para que nada tenga probabilidad 0.
- **Muestreo**: elegir al azar respetando las probabilidades.

## Habilidades de programacion

- Diccionarios como tablas hash.
- Tuplas como claves.
- `.get(clave, por_defecto)` para evitar `KeyError`.
- Comprensiones de listas y diccionarios.
- `random.choices` con pesos.
- `random.seed` para reproducibilidad.
- `json.dump` y `json.load` para persistencia.
- `assert` para pruebas unitarias.

## Limitaciones del modelo

- Solo mira **una** palabra atras, lo que produce frases gramaticalmente raras.
- No entiende significado, solo estadistica.
- Cuanto mas grande el corpus, mejor funciona.

## Y ahora que

Ideas para seguir ampliando:

1. **Trigramas**: mirar dos palabras atras en lugar de una.
2. **`<EOS>`**: anadir un token de fin de oracion para parar solo.
3. **Interfaz interactiva**: pedir al usuario una semilla con `input()`.
4. **Corpus real**: usar un fragmento de un libro grande.
5. **Comparar con HuggingFace**: ver que hace un tokenizador profesional.

Ya has construido un mini-modelo de lenguaje desde cero.